# Optimization Experiments — 2×2×2 Rubik's Cube

This notebook runs all experiments for the numerical optimization final report.
Results are cached to `results/` as JSON so training runs are never repeated unnecessarily.
Figures are saved to `figures/` as PDFs for inclusion in `report.tex`.

**Experiments:**
1. Optimizer comparison (AdamW vs Adam vs SGD with momentum)
2. Learning rate schedule ablation (cosine annealing vs step decay vs constant)
3. Hyperparameter sensitivity (d_model sweep, weight decay sweep)
4. Learned heuristic quality (admissibility, MAE by distance, per-distance accuracy)
5. Loss landscape (filter-normalized, Li et al. 2018)

In [ ]:
import sys
import json
import time
from pathlib import Path
from collections import defaultdict

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 130,
})

from dataset import load_split
from model import CubeTransformer

RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Device: {device}")

In [ ]:
DATA_DIR = ROOT / "data"

def make_loader(data, batch_size, shuffle):
    ds = TensorDataset(
        torch.from_numpy(data["states"].astype("float32")),
        torch.from_numpy(data["optimal_distance"].astype("int64")),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

train_data = load_split(DATA_DIR / "train.npz")
val_data   = load_split(DATA_DIR / "val.npz")

BATCH_SIZE = 512
train_loader = make_loader(train_data, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(val_data,   BATCH_SIZE, shuffle=False)

# Inverse-frequency class weights (same as train.py)
train_labels = torch.from_numpy(train_data["optimal_distance"].astype("int64"))
counts = torch.bincount(train_labels, minlength=12).float()
class_weights = (1.0 / (counts + 1))
class_weights = class_weights / class_weights.sum() * 12

print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}")
print(f"Class weights: min={class_weights.min():.2f}  max={class_weights.max():.2f}")

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, class_weights):
    """Returns (val_loss, val_acc, depth_acc, depth_mae, depth_pred_errors)."""
    model.eval()
    cw = class_weights.to(device)
    total_loss = total_correct = total = 0
    depth_preds = defaultdict(list)

    for states, targets in loader:
        states, targets = states.to(device), targets.to(device)
        logits = model(states)
        loss   = F.cross_entropy(logits, targets, weight=cw, reduction="sum")
        preds  = logits.argmax(dim=-1)

        total_loss    += loss.item()
        total_correct += (preds == targets).sum().item()
        total         += len(targets)

        for t, p in zip(targets.cpu().numpy(), preds.cpu().numpy()):
            depth_preds[int(t)].append(int(p))

    acc  = total_correct / total
    loss = total_loss / total
    depth_acc = {d: float(np.mean(np.array(depth_preds[d]) == d))   for d in sorted(depth_preds)}
    depth_mae = {d: float(np.mean(np.abs(np.array(depth_preds[d]) - d))) for d in sorted(depth_preds)}
    depth_errors = {d: (np.array(depth_preds[d]) - d).tolist() for d in sorted(depth_preds)}

    return loss, acc, depth_acc, depth_mae, depth_errors

In [ ]:
def run_experiment(
    name: str,
    optimizer_type: str = "adamw",   # "adamw", "adam", "sgd"
    scheduler_type: str = "cosine",  # "cosine", "constant", "step"
    d_model: int = 128,
    n_layers: int = 4,
    n_heads: int = 4,
    lr: float = 1e-3,
    wd: float = 1e-4,
    epochs: int = 30,
    force: bool = False,
) -> list:
    """Train a model and return history. Caches to results/{name}.json."""
    out_path = RESULTS_DIR / f"{name}.json"
    if out_path.exists() and not force:
        print(f"  [cached]  {name}")
        return json.loads(out_path.read_text())

    print(f"  [running] {name}  ({optimizer_type}, {scheduler_type}, d={d_model}, lr={lr:.0e}, wd={wd:.0e})")
    model = CubeTransformer(
        d_model=d_model, n_layers=n_layers, n_heads=n_heads, n_classes=12
    ).to(device)

    # --- Optimizer ---
    if optimizer_type == "adamw":
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    elif optimizer_type == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif optimizer_type == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=0.9)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_type}")

    # --- Scheduler ---
    if scheduler_type == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    elif scheduler_type == "constant":
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda e: 1.0)
    elif scheduler_type == "step":
        sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.3)
    else:
        raise ValueError(f"Unknown scheduler: {scheduler_type}")

    cw = class_weights.to(device)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.perf_counter()
        running_loss = 0.0

        for states, targets in train_loader:
            states, targets = states.to(device), targets.to(device)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(states), targets, weight=cw)
            loss.backward()
            opt.step()
            running_loss += loss.item()

        sched.step()
        train_loss = running_loss / len(train_loader)
        val_loss, val_acc, depth_acc, depth_mae, depth_errors = evaluate(
            model, val_loader, device, class_weights
        )
        elapsed = time.perf_counter() - t0
        current_lr = sched.get_last_lr()[0]

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "depth_acc": depth_acc,
            "depth_mae": depth_mae,
            "depth_errors": depth_errors,
            "lr": current_lr,
            "elapsed": elapsed,
        })

        if epoch % 10 == 0 or epoch == epochs:
            print(f"    epoch {epoch:3d}  val_acc={val_acc*100:.1f}%  loss={val_loss:.4f}  lr={current_lr:.2e}  ({elapsed:.1f}s)")

    out_path.write_text(json.dumps(history, indent=2))
    print(f"  → saved {out_path}")
    return history

---
## 1. Optimizer Comparison

Compare AdamW, Adam, and SGD with momentum under cosine annealing.
SGD uses a higher learning rate (0.05) selected by short grid search;
AdamW and Adam use lr=1e-3.

In [ ]:
print("=== 1. Optimizer Comparison ===")
results_opt = {
    "adamw": run_experiment("opt_adamw", optimizer_type="adamw", scheduler_type="cosine", lr=1e-3),
    "adam":  run_experiment("opt_adam",  optimizer_type="adam",  scheduler_type="cosine", lr=1e-3),
    "sgd":   run_experiment("opt_sgd",   optimizer_type="sgd",   scheduler_type="cosine", lr=0.05),
}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
colors = {"adamw": "#1f77b4", "adam": "#ff7f0e", "sgd": "#2ca02c"}
labels = {"adamw": "AdamW", "adam": "Adam", "sgd": "SGD + momentum"}

for key, hist in results_opt.items():
    epochs   = [h["epoch"]       for h in hist]
    val_acc  = [h["val_acc"]*100 for h in hist]
    val_loss = [h["val_loss"]    for h in hist]
    c = colors[key]
    axes[0].plot(epochs, val_acc,  color=c, label=labels[key], linewidth=1.8)
    axes[1].plot(epochs, val_loss, color=c, label=labels[key], linewidth=1.8)

for ax, ylabel, title in zip(axes,
    ["Validation Accuracy (%)", "Validation Loss"],
    ["Accuracy by Optimizer",   "Loss by Optimizer"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "optimizer_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved optimizer_comparison.pdf")

---
## 2. Learning Rate Schedule Ablation

Hold optimizer fixed to AdamW (lr=1e-3) and compare three schedules:
cosine annealing, step decay (×0.3 every 10 epochs), and constant.

In [ ]:
print("=== 2. LR Schedule Ablation ===")
results_sched = {
    "cosine":   run_experiment("sched_cosine",   optimizer_type="adamw", scheduler_type="cosine",   lr=1e-3),
    "step":     run_experiment("sched_step",     optimizer_type="adamw", scheduler_type="step",     lr=1e-3),
    "constant": run_experiment("sched_constant", optimizer_type="adamw", scheduler_type="constant", lr=1e-3),
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
colors_s = {"cosine": "#1f77b4", "step": "#d62728", "constant": "#9467bd"}
labels_s = {"cosine": "Cosine annealing", "step": "Step decay", "constant": "Constant"}

for key, hist in results_sched.items():
    epochs   = [h["epoch"]       for h in hist]
    val_acc  = [h["val_acc"]*100 for h in hist]
    val_loss = [h["val_loss"]    for h in hist]
    lrs      = [h["lr"]          for h in hist]
    c = colors_s[key]
    axes[0].plot(epochs, val_acc,  color=c, label=labels_s[key], linewidth=1.8)
    axes[1].plot(epochs, val_loss, color=c, label=labels_s[key], linewidth=1.8)
    axes[2].plot(epochs, lrs,      color=c, label=labels_s[key], linewidth=1.8)

for ax, ylabel, title in zip(axes,
    ["Val Accuracy (%)", "Val Loss", "Learning Rate"],
    ["Accuracy",         "Loss",     "LR Trajectory"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "schedule_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved schedule_comparison.pdf")

---
## 3. Hyperparameter Sensitivity

Two sweeps, all under AdamW + cosine annealing:
- **Width sweep**: d_model ∈ {64, 128, 256} (default n_layers=4, n_heads=4)
- **Weight decay sweep**: λ ∈ {0, 1e-4, 1e-3, 1e-2} (default d_model=128)

In [ ]:
print("=== 3. Hyperparameter Sensitivity ===")

# Width sweep
d_models = [64, 128, 256]
results_dmodel = {}
for d in d_models:
    results_dmodel[d] = run_experiment(
        f"dmodel_{d}", optimizer_type="adamw", scheduler_type="cosine",
        d_model=d, lr=1e-3, wd=1e-4,
    )

# Weight decay sweep (reuse d_model=128 result from opt_adamw above)
wd_values = [0.0, 1e-4, 1e-3, 1e-2]
results_wd = {}
for wd in wd_values:
    name = f"wd_{wd:.0e}" if wd > 0 else "wd_0"
    results_wd[wd] = run_experiment(
        name, optimizer_type="adamw", scheduler_type="cosine",
        d_model=128, lr=1e-3, wd=wd,
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Width sweep: convergence curves
cmap_d = plt.cm.Blues
for i, (d, hist) in enumerate(sorted(results_dmodel.items())):
    c = cmap_d(0.4 + 0.3 * i)
    epochs  = [h["epoch"]       for h in hist]
    val_acc = [h["val_acc"]*100 for h in hist]
    axes[0].plot(epochs, val_acc, color=c, label=f"d={d}", linewidth=1.8)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Val Accuracy (%)")
axes[0].set_title("Width Sensitivity")
axes[0].legend()

# Weight decay sweep: convergence curves
cmap_w = plt.cm.Reds
wd_labels = {0.0: "0 (none)", 1e-4: "1e-4", 1e-3: "1e-3", 1e-2: "1e-2"}
for i, (wd, hist) in enumerate(sorted(results_wd.items())):
    c = cmap_w(0.3 + 0.18 * i)
    epochs  = [h["epoch"]       for h in hist]
    val_acc = [h["val_acc"]*100 for h in hist]
    axes[1].plot(epochs, val_acc, color=c, label=f"λ={wd_labels[wd]}", linewidth=1.8)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val Accuracy (%)")
axes[1].set_title("Weight Decay Sensitivity")
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "hyperparameter_sensitivity.pdf", bbox_inches="tight")
plt.show()
print("Saved hyperparameter_sensitivity.pdf")

---
## 4. Learned Heuristic Quality

Load the best AdamW + cosine model (from opt_adamw results) and evaluate:
- Per-distance accuracy and MAE
- Admissibility: fraction of states where model overestimates distance
- Distribution of prediction errors

In [ ]:
print("=== 4. Learned Heuristic Quality ===")

# Re-train the best model (or load if already done)
# We need the actual model object, not just history — retrain from the cached run
# to ensure determinism. Alternatively save model weights; for now retrain with fixed seed.

torch.manual_seed(0)
best_model = CubeTransformer(d_model=128, n_layers=4, n_heads=4, n_classes=12).to(device)
best_opt   = torch.optim.AdamW(best_model.parameters(), lr=1e-3, weight_decay=1e-4)
best_sched = torch.optim.lr_scheduler.CosineAnnealingLR(best_opt, T_max=30, eta_min=1e-3/20)
cw = class_weights.to(device)

for epoch in range(1, 31):
    best_model.train()
    for states, targets in train_loader:
        states, targets = states.to(device), targets.to(device)
        best_opt.zero_grad(set_to_none=True)
        F.cross_entropy(best_model(states), targets, weight=cw).backward()
        best_opt.step()
    best_sched.step()
    if epoch % 10 == 0:
        _, acc, _, _, _ = evaluate(best_model, val_loader, device, class_weights)
        print(f"  epoch {epoch}  val_acc={acc*100:.1f}%")

_, final_acc, depth_acc, depth_mae, depth_errors = evaluate(
    best_model, val_loader, device, class_weights
)
print(f"\nFinal val accuracy: {final_acc*100:.1f}%")

In [ ]:
distances = sorted(depth_acc.keys())

# Admissibility: fraction of predictions that overestimate (pred > true)
overestimate_frac = {
    d: float(np.mean(np.array(depth_errors[d]) > 0))
    for d in distances
}
total_overestimate = np.mean([
    v for errs in depth_errors.values() for v in errs if v > 0
] and [1] or [0])  # fallback
all_errors = [v for errs in depth_errors.values() for v in errs]
overestimate_pct = 100 * np.mean(np.array(all_errors) > 0)
print(f"Overall overestimate rate: {overestimate_pct:.1f}%  ({'NOT ' if overestimate_pct > 0 else ''}admissible)")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# Per-distance accuracy
axes[0].bar(distances, [depth_acc[d]*100 for d in distances], color="#1f77b4", alpha=0.85)
axes[0].axhline(100/12, color="gray", linestyle="--", linewidth=1, label="Random (8.3%)")
axes[0].set_xlabel("True Distance")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Per-Distance Accuracy")
axes[0].legend(fontsize=8)

# Per-distance MAE
axes[1].bar(distances, [depth_mae[d] for d in distances], color="#ff7f0e", alpha=0.85)
axes[1].set_xlabel("True Distance")
axes[1].set_ylabel("Mean Absolute Error")
axes[1].set_title("Heuristic MAE by Distance")

# Error distribution
error_vals = np.array(all_errors)
bins = np.arange(error_vals.min() - 0.5, error_vals.max() + 1.5)
axes[2].hist(error_vals, bins=bins, color="#2ca02c", alpha=0.8, edgecolor="white")
axes[2].axvline(0, color="black", linewidth=1.2, linestyle="--")
axes[2].set_xlabel("Prediction Error (ĥ − d*)")
axes[2].set_ylabel("Count")
axes[2].set_title("Error Distribution")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "heuristic_quality.pdf", bbox_inches="tight")
plt.show()
print("Saved heuristic_quality.pdf")

---
## 5. Loss Landscape

Filter-normalized 2D loss surface around the trained AdamW solution.
Following Li et al. (2018): two random directions in parameter space,
each normalized so each filter has the same Frobenius norm as the
corresponding filter in the trained weights.

**Note:** This cell evaluates loss at a 21×21 grid of perturbation values.
With the small model (~200k params) and a subset of the val set it takes
approximately 5–10 minutes.

In [ ]:
def filter_normalize(direction, reference):
    """Scale each filter in direction to match the Frobenius norm of reference."""
    out = []
    for d, w in zip(direction, reference):
        if w.dim() > 1:
            w_norms = w.view(w.shape[0], -1).norm(dim=1, keepdim=True)
            d_norms = d.view(d.shape[0], -1).norm(dim=1, keepdim=True) + 1e-10
            scale = (w_norms / d_norms).view(-1, *([1] * (w.dim() - 1)))
            out.append(d * scale)
        else:
            out.append(d * (w.norm() / (d.norm() + 1e-10)))
    return out


def compute_loss_landscape(model, loader, device, class_weights,
                           grid_size=21, alpha_range=1.0, seed=42):
    """Evaluate loss on a 2D perturbation grid around model weights."""
    weights = [p.data.clone() for p in model.parameters()]
    cw = class_weights.to(device)

    torch.manual_seed(seed)
    dir1 = filter_normalize([torch.randn_like(w) for w in weights], weights)
    dir2 = filter_normalize([torch.randn_like(w) for w in weights], weights)

    alphas = np.linspace(-alpha_range, alpha_range, grid_size)
    betas  = np.linspace(-alpha_range, alpha_range, grid_size)
    loss_grid = np.zeros((grid_size, grid_size))

    for i, alpha in enumerate(alphas):
        for j, beta in enumerate(betas):
            for p, w, d1, d2 in zip(model.parameters(), weights, dir1, dir2):
                p.data.copy_(w + alpha * d1 + beta * d2)
            total_loss = total = 0
            with torch.no_grad():
                for states, targets in loader:
                    states, targets = states.to(device), targets.to(device)
                    loss = F.cross_entropy(model(states), targets, weight=cw, reduction="sum")
                    total_loss += loss.item()
                    total      += len(targets)
            loss_grid[i, j] = total_loss / total
        if (i + 1) % 5 == 0:
            print(f"  Row {i+1}/{grid_size} done")

    # Restore original weights
    for p, w in zip(model.parameters(), weights):
        p.data.copy_(w)

    return alphas, betas, loss_grid


print("Computing loss landscape (this takes ~5-10 min)...")
alphas, betas, loss_grid = compute_loss_landscape(
    best_model, val_loader, device, class_weights,
    grid_size=21, alpha_range=1.0,
)
np.savez(RESULTS_DIR / "loss_landscape.npz",
         alphas=alphas, betas=betas, loss_grid=loss_grid)
print("Done. Saved loss_landscape.npz")

In [ ]:
# Load cached landscape if available
landscape = np.load(RESULTS_DIR / "loss_landscape.npz")
alphas, betas, loss_grid = landscape["alphas"], landscape["betas"], landscape["loss_grid"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Contour plot
A, B = np.meshgrid(alphas, betas, indexing="ij")
center_loss = loss_grid[len(alphas)//2, len(betas)//2]
levels = np.linspace(loss_grid.min(), min(loss_grid.max(), center_loss * 5), 20)
cf = axes[0].contourf(A, B, loss_grid, levels=levels, cmap="viridis")
axes[0].contour(A, B, loss_grid, levels=levels, colors="white", linewidths=0.4, alpha=0.4)
axes[0].scatter([0], [0], color="red", s=60, zorder=5, label=r"$\theta^*$")
fig.colorbar(cf, ax=axes[0], label="Val Loss")
axes[0].set_xlabel(r"$\alpha$ (direction 1)")
axes[0].set_ylabel(r"$\beta$ (direction 2)")
axes[0].set_title("Loss Landscape (contour)")
axes[0].legend(fontsize=9)

# 1D slices through center
mid = len(alphas) // 2
axes[1].plot(alphas, loss_grid[:, mid], color="#1f77b4", label="direction 1 (β=0)", linewidth=1.8)
axes[1].plot(betas,  loss_grid[mid, :], color="#ff7f0e", label="direction 2 (α=0)", linewidth=1.8)
axes[1].axvline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_xlabel("Perturbation magnitude")
axes[1].set_ylabel("Val Loss")
axes[1].set_title("1D Slices Through θ*")
axes[1].legend()

fig.suptitle("Filter-Normalized Loss Landscape", fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "loss_landscape.pdf", bbox_inches="tight")
plt.show()
print("Saved loss_landscape.pdf")

---
## Summary of Results

Run this cell after all experiments complete to print a summary table.

In [ ]:
print(f"{'Experiment':<35} {'Final Val Acc':>14} {'Best Val Acc':>13}")
print("-" * 64)

all_results = {
    "AdamW (cosine)": results_opt["adamw"],
    "Adam (cosine)": results_opt["adam"],
    "SGD+mom (cosine)": results_opt["sgd"],
    "AdamW (step)": results_sched["step"],
    "AdamW (constant)": results_sched["constant"],
    **{f"d_model={d}": h for d, h in results_dmodel.items() if d != 128},
    **{f"wd={wd:.0e}": h for wd, h in results_wd.items()},
}

for name, hist in all_results.items():
    final = hist[-1]["val_acc"] * 100
    best  = max(h["val_acc"] for h in hist) * 100
    print(f"{name:<35} {final:>13.1f}%  {best:>12.1f}%")